# Binning problems

Three **assignment-matrix** solvers on `ortidy`'s bundled data. Each returns a `SolveResult`: the result `frame` (same dataframe backend you passed in) plus `status`, `objective`, and `metadata`.

In [ ]:
import pandas as pd

import ortidy
from ortidy import bin_packing, knapsack, multi_knapsack

pd.set_option("display.max_rows", 500)

## 1. Knapsack

Maximize the value of items packed into a single capacity-limited basket. `knapsack` adds an `isIncluded` boolean column. See https://developers.google.com/optimization/bin/knapsack

In [ ]:
items = ortidy.data.items_knapsack()
result = knapsack(items, capacity=850)

print(result.status, "| total value:", result.objective)
result.frame.groupby("isIncluded").sum()

## 2. Multiple knapsack

Pack items into several baskets of different capacities, maximizing total packed value. Each item gets a `binId` (or `NaN` if left unpacked). See https://developers.google.com/optimization/bin/multiple_knapsack

In [ ]:
items = ortidy.data.items_multi()
bins = ortidy.data.bins()
items["itemId"] = items.index
bins["binId"] = bins.index

result = multi_knapsack(items=items, bins=bins, item_id="itemId")
print(result.status, "| packed value:", result.objective)

packed = (
    result.frame.dropna(subset=["binId"])
    .astype({"binId": int})
    .merge(bins, on="binId")
)
packed.groupby("binId").agg(
    value=("value", "sum"),
    weight=("weight", "sum"),
    capacity=("capacity", "mean"),
    items=("itemId", list),
)

In [ ]:
result.frame.dropna(subset=["binId"])[["weight", "value"]].sum()

## 3. Bin packing

Pack every item into equal-capacity bins, minimizing the number of bins used. See https://developers.google.com/optimization/bin/bin_packing

In [ ]:
items = ortidy.data.items_bin_packing()
items["itemId"] = items.index

result = bin_packing(items, capacity=100, item_id="itemId")
print(result.status, "| bins used:", result.objective)

result.frame.groupby("binId").agg(
    weight=("weight", "sum"),
    weights=("weight", list),
    items=("itemId", list),
)

In [ ]:
print("Number of bins required to include all items:")
result.frame["binId"].nunique()

## 4. Multidimensional knapsack

Items constrained by **weight *and* volume** — pass a list of weight columns and a list of capacities.

In [ ]:
items = pd.DataFrame({
    "value": [60, 100, 120, 40],
    "weight": [10, 20, 30, 15],
    "volume": [2, 3, 4, 1],
})
result = knapsack(items, capacity=[50, 6], weight=["weight", "volume"])
print(result.status, "| value:", result.objective)
result.frame